## Gaussian Elimination

In this section we develop Python code to solve linear systems in the most direct way, via **Gaussian Elimination** (or just **elimination**).

**The main concept of elimination:**

* transform the given system to an equivalent system (with same solution) that but that is *easier* to solve.

The transformation comes in the form of a sequence of **row operations** that preserve the solution of the system while making the solution readily available. There are three row operations that are helpful:

1. Exchange the position of two rows/equations.
    * Notation: $R_i \leftrightarrow{} R_j$
4. Multiply a row/equation by any nonzero number:
    * Notation: $R_i \to k R_i$
6. Replace any row/equation with the sum of itself and a multiple of another row/equation.
   * Notation: $R_i \to R_i + k R_j$



<a id='GE1'></a>
### Example:  Row operations and elimination

Let's look at an example.

$$
\begin{eqnarray*}
x + 2 y - z & = & 3\\
2x + y - 2z & = & 3\\
-3x + y + z & = & -6
\end{eqnarray*}
$$

Written in augmented form:

$$
\begin{equation}
\left[ \begin{array}{rrr|r}
1 & 2 & -1 & 3 \\
2 & 1 & -2 & 3 \\
-3 & 1 & 1 & -6
\end{array}\right]
\end{equation}
$$

Row operations to transform to an equivalent (upper triangular matrix) are

1. $R_2 \to R_2 - 2 R_1$
2. $R_3 \to R_3 + 3 R_1$
3. $R_3 \to R_3 + \dfrac{7}{3} R_2$

and results in the follwoing equivalent system

$$
\begin{equation}
\left[ \begin{array}{rrr|r}
1 & 2 & -1 & 3 \\
0 & -3 & 0 & -3 \\
0 & 0 & -2 & -4
\end{array}\right]
\end{equation}
$$

At this point, back-substitution can be used to arrive at the solution. Rather than do this now, let's build a python routine to execute the elimination process, illustrated above, to a more general $n\times n$ system.  

We'll assign the array the name $A$, so that we can refer to it later.

In [25]:
import sys
import numpy as np
# Define the augmented matrix in the above example.
# We'll use this as a test case for working with nxn systems.
A = np.array([[1,2,-1,3],[2,1,-2,3],[-3,1,1,-6]])
print(A)

[[ 1  2 -1  3]
 [ 2  1 -2  3]
 [-3  1  1 -6]]


We could start performing operations on our array, but instead we will first write a few bits of code that will do each of these operations in a systematic manner.


The goal of elimination is to perform row operations to produce a an equivalent upper-triangular matrix:

$$
\begin{equation}
\left[ \begin{array}{cccc} 1 & * & * & * \\ 0 & 1 & * & * \\ 0 & 0 & 1 & * \end{array}\right]
\end{equation}
$$

where the asterisk denote some entry (may or may not be zero). Let's now generalize this to an $n\times n$ system:  

In [26]:
import sys
import numpy as np
# Define the augmented matrix in the above example.
# We'll use this as a test case for working with nxn systems.
A = np.array([[1,2,-1,3],[2,1,-2,3],[-3,1,1,-6]])
# print(A)

def elimin(A,b):
# =============================================================================
#     A is an nxn matrix .
#     b is the right-hand-side of Ax = b.
#     indicies counting from 1. Note python uses zero-based indexing
# =============================================================================
    m = A.shape[0]  # m is number of rows in A
    n = A.shape[1]  # n is number of columns in A

    if m != n:
        # PRINT AN ERROR MESSAGE
        print("Matrix A must be square.")
        return

    x = np.zeros(n)

    B = np.copy(A).astype('float64')
    r = np.copy(b).astype('float64')
    eps = sys.float_info.epsilon

    # Combine B and r into an augmented matrix for elimination
    augmented_matrix = np.hstack((B, r.reshape(-1, 1)))

    # visit each column, up to j = n-1
    for j in range(n):
        if abs(augmented_matrix[j][j]) < eps: # CONDITION TO IDENTIFY ZERO PIVOT
            print("Zero pivot encountered.")
            return augmented_matrix, x # Return current state

        # visit each row, from i = j+1 to n
        for i in range(j+1,n):
            scale = augmented_matrix[i][j] / augmented_matrix[j][j] # STORE ROW MULTIPLYER
            # visit each column in row i, from k = j to n (inclusive of the augmented part)
            for k in range(j,n+1): # Iterate up to n+1 to include the augmented column
                augmented_matrix[i][k]  = augmented_matrix[i][k] - scale * augmented_matrix[j][k] # AN APPROPRIATE ROW OPERATION


    # Solve using back-substitution
    # Visit each row from bottom up
    for i in range(n-1,-1,-1):
        sum_jx = 0
        for j in range(i+1,n):
            sum_jx += augmented_matrix[i][j] * x[j]
        x[i] = (augmented_matrix[i][n] - sum_jx) / augmented_matrix[i][i]


    return augmented_matrix, x # Return the full augmented matrix and the solution vector

In [27]:
elimin(A[0:3,0:3],A[0:3,3])

(array([[ 1.,  2., -1.,  3.],
        [ 0., -3.,  0., -3.],
        [ 0.,  0., -2., -4.]]),
 array([3., 1., 2.]))

(a) Validate your code by solving the following systems Ax = b, for the given A and b. You should check you rresults with those found by hand.

i. A =
$\begin{bmatrix}
2 & -3 \\
5 & -6
\end{bmatrix}
$
, b =
$\begin{bmatrix}
2 \\
8
\end{bmatrix}$

In [29]:
A = np.array([[2, -3], [5, -6]])
b = np.array([2, 8])

print("Matrix A:\n", A)
print("\nVector b:\n", b)

# Combine A and b
augmented_matrix_initial = np.hstack((A, b.reshape(-1, 1)))
print("\nAugmented Matrix [A|b]:\n", augmented_matrix_initial)

elim_augmented_matrix, solution = elimin(A, b)

print("\nMatrix after elimination:\n", elim_augmented_matrix)
print("\nSolution vector:\n", solution)

Matrix A:
 [[ 2 -3]
 [ 5 -6]]

Vector b:
 [2 8]

Augmented Matrix [A|b]:
 [[ 2 -3  2]
 [ 5 -6  8]]

Matrix after elimination:
 [[ 2.  -3.   2. ]
 [ 0.   1.5  3. ]]

Solution vector:
 [4. 2.]


ii. A =
$\begin{bmatrix}
1 & 2 & -1 \\
0 & 3 & 1 \\
2 & -1 & 1
\end{bmatrix}$
, b =
$\begin{bmatrix}
2 \\
4 \\
2
\end{bmatrix}$

In [30]:
A = np.array([[1, 2, -1], [0, 3, 1], [2, -1, 1]])
b = np.array([2, 4, 2])

print("Matrix A:\n", A)
print("\nVector b:\n", b)

# Combine A and b
augmented_matrix = np.hstack((A, b.reshape(-1, 1)))
print("\nAugmented Matrix [A|b]:\n", augmented_matrix)

elim_result, solution = elimin(A, b)

print("\nMatrix after elimination:\n", elim_result)
print("\nSolution vector:\n", solution)

Matrix A:
 [[ 1  2 -1]
 [ 0  3  1]
 [ 2 -1  1]]

Vector b:
 [2 4 2]

Augmented Matrix [A|b]:
 [[ 1  2 -1  2]
 [ 0  3  1  4]
 [ 2 -1  1  2]]

Matrix after elimination:
 [[ 1.          2.         -1.          2.        ]
 [ 0.          3.          1.          4.        ]
 [ 0.          0.          4.66666667  4.66666667]]

Solution vector:
 [1. 1. 1.]


(b) Use your code to solve Ax = b with A and b given below. No need to solve by hand if you've validated your code!

i. A =
$\begin{bmatrix}
1 & -1 & 1 & 2 \\
0 & 2 & 1 & 0 \\
1 & 3 & 4 & 4 \\
4 & 2 & 1 & -1
\end{bmatrix}
$
, b =
$\begin{bmatrix}
1 \\
2 \\
3 \\
4 \\
\end{bmatrix}$

In [31]:
A = np.array([[1, -1, 1, 2], [0, 2, 1, 0], [1, 3, 4, 4], [4, 2, 1, -1]])
b = np.array([1, 2, 3, 4])

print("Matrix A:\n", A)
print("\nVector b:\n", b)

# Combine A and b
augmented_matrix = np.hstack((A, b.reshape(-1, 1)))
print("\nAugmented Matrix [A|b]:\n", augmented_matrix)

elim_result, solution = elimin(A, b)

print("\nMatrix after elimination:\n", elim_result)
print("\nSolution vector:\n", solution)

Matrix A:
 [[ 1 -1  1  2]
 [ 0  2  1  0]
 [ 1  3  4  4]
 [ 4  2  1 -1]]

Vector b:
 [1 2 3 4]

Augmented Matrix [A|b]:
 [[ 1 -1  1  2  1]
 [ 0  2  1  0  2]
 [ 1  3  4  4  3]
 [ 4  2  1 -1  4]]

Matrix after elimination:
 [[  1.  -1.   1.   2.   1.]
 [  0.   2.   1.   0.   2.]
 [  0.   0.   1.   2.  -2.]
 [  0.   0.   0.   3. -18.]]

Solution vector:
 [-1. -4. 10. -6.]


6. **Programming** *LU* **Decomposition:**
Use the Jupyter Notebook and the code you developed in problem 3 for Gaussian Elimination to write a Python function to take a matrix A as input and output *L* and *U*.

In [32]:
import numpy as np

def lu_decompose(A):
    """
    Performs LU decomposition on a square matrix A.

    Args:
        A: A square NumPy array.

    Returns:
        A tuple containing:
            L: The lower triangular matrix.
            U: The upper triangular matrix.
        Returns None, None if decomposition is not possible (e.g., zero pivot).
    """
    n = A.shape[0]
    if A.shape[1] != n:
        print("Matrix A must be square.")
        return None, None

    U = np.copy(A).astype('float64')
    L = np.eye(n, dtype='float64') # Initialize L as identity matrix

    for j in range(n):
        if np.abs(U[j][j]) < 1e-10: # Check for zero pivot
            print("Zero pivot encountered. LU decomposition not possible.")
            return None, None

        for i in range(j + 1, n):
            multiplier = U[i][j] / U[j][j]
            L[i][j] = multiplier # Store the multiplier in L
            U[i, j:] = U[i, j:] - multiplier * U[j, j:] # Perform row operation on U

    return L, U

Testing out code on problem 4.

(a) $\begin{bmatrix}
1 & 2 \\
3 & 4
\end{bmatrix}$

In [33]:
A = np.array([[1, 2], [3, 4]])
L, U = lu_decompose(A)

print("Matrix A:\n", A)
print("\nLower Triangular Matrix L:\n", L)
print("\nUpper Triangular Matrix U:\n", U)

Matrix A:
 [[1 2]
 [3 4]]

Lower Triangular Matrix L:
 [[1. 0.]
 [3. 1.]]

Upper Triangular Matrix U:
 [[ 1.  2.]
 [ 0. -2.]]


(b) $\begin{bmatrix}
3 & -4 \\
-5 & 2
\end{bmatrix}$

In [34]:
A = np.array([[3, -4], [-5, 2]])
L, U = lu_decompose(A)

print("Matrix A:\n", A)
print("\nLower Triangular Matrix L:\n", L)
print("\nUpper Triangular Matrix U:\n", U)

Matrix A:
 [[ 3 -4]
 [-5  2]]

Lower Triangular Matrix L:
 [[ 1.          0.        ]
 [-1.66666667  1.        ]]

Upper Triangular Matrix U:
 [[ 3.         -4.        ]
 [ 0.         -4.66666667]]


(c) $\begin{bmatrix}
4 & 2 & 0 \\
4 & 4 & 2 \\
2 & 2 & 3
\end{bmatrix}$

In [37]:
A = np.array([[4, 2, 0], [4, 4, 2], [2, 2, 3]])
L, U = lu_decompose(A)

print("Matrix A:\n", A)
print("\nLower Triangular Matrix L:\n", L)
print("\nUpper Triangular Matrix U:\n", U)

Matrix A:
 [[4 2 0]
 [4 4 2]
 [2 2 3]]

Lower Triangular Matrix L:
 [[1.  0.  0. ]
 [1.  1.  0. ]
 [0.5 0.5 1. ]]

Upper Triangular Matrix U:
 [[4. 2. 0.]
 [0. 2. 2.]
 [0. 0. 2.]]
